# Notebook 4: Score pages with a fine-tuned model

Use this right after training to see how well your model reads. It loads a run from Notebook 2
**once** and keeps it ready, so after the first couple of minutes each page takes only seconds.
You can:

- **score the dev pages** (or any pages with a known transcription) with the competition's own
  scoring code, `metric.py`, the same code Kaggle uses;
- **compare with the model before training** (the Kurrent adapter on its own), page by page;
- **look at any page**: its image, the correct text and the model's text, with the differences
  coloured in;
- **read any page**, including test pages, which have no correct text to score against.

**Needs a GPU**, like Notebook 2. Run the cells from top to bottom with **Shift + Enter**.
Kernel: **conda_pytorch_p310** (or whichever `conda_pytorch` kernel your instance lists).

## Step 1. Install the libraries

The same as Notebook 2, Step 1: needed after every start, and before any other cell.

In [ ]:
%pip install -q -r requirements-training.txt

import importlib.util
if importlib.util.find_spec("torch") is None:
    raise SystemExit("This kernel has no PyTorch, so it can't run the model. Choose Kernel → Change Kernel, "
                     "pick conda_pytorch_p310 (or another conda_pytorch kernel), then run this cell again.")

## Step 2. Set up the folders

In [ ]:
import importlib.util
import os
import subprocess
import sys

# Which kernel is this? It should be conda_pytorch_p310 (or another conda_pytorch kernel).
kernel = os.path.basename(os.path.dirname(os.path.dirname(sys.executable)))
print("Kernel environment:", kernel)
# The notebooks need Pillow (images) and pandas (tables) in this kernel. If it
# doesn't have them, install them here.
missing = [pkg for module, pkg in (("PIL", "pillow"), ("pandas", "pandas"))
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing", ", ".join(missing), "into this kernel ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Everything this project downloads or creates lives here. It is on the
# notebook instance's storage volume, so it survives Stop and Start.
WORK = "/home/ec2-user/SageMaker/badger-finetune"
DATA = os.path.join(WORK, "data")            # page images, labels, line images
RUNS = os.path.join(WORK, "runs")            # one folder per training run
SCRIPTS = os.path.abspath("scripts")         # the scripts folder next to this notebook
for folder in (WORK, DATA, RUNS):
    os.makedirs(folder, exist_ok=True)
sys.path.insert(0, SCRIPTS)

# Keep the 9 GB model download on the big volume, not the small system disk.
os.environ["HF_HOME"] = os.path.join(WORK, "hf_cache")

SHARED_BUCKET = "ml-marathon-shared"   # the class bucket with the Badger Scribe data
SHARED_PREFIX = "badger-scribe"
print("Working folder:", WORK)

# The competition's scoring code: metric.py at the top of the repository.
REPO = os.path.abspath("..")
sys.path.insert(0, REPO)
try:
    import metric
    page_cer = metric.page_cer
    print("Scoring with", os.path.join(REPO, "metric.py"))
except ImportError:
    from kurrent_common import page_cer   # a copy of the same calculation
    print("metric.py not found; scoring with the copy in scripts/kurrent_common.py (same numbers)")

## Step 3. Pick the run to test

This lists your finished runs, newest first, with the dev score each one reported in Notebook 2.
**The newest one is picked**, so right after training you don't need to change anything. To test a
different run, type its name.

Only runs trained on the same kind of images that Notebook 1 cut last (lines, groups or pages)
can be tested, because the model has to read the same kind of image it learned from.

In [ ]:
import json
import pandas as pd

crop = json.load(open(os.path.join(DATA, "kade_lines", "crop_settings.json")))
runs = []
for name in os.listdir(RUNS):
    path = os.path.join(RUNS, name, "model", "train_summary.json")
    if os.path.exists(path):
        s = json.load(open(path))
        runs.append({"run": name, "finished": pd.Timestamp(os.path.getmtime(path), unit="s").floor("min"),
                     "cut as": s.get("crop_mode", "lines"), "dev CER before": s["dev_cer_before"],
                     "dev CER best": s["dev_cer_best"], "dev pages": s["dev_pages"]})
runs = pd.DataFrame(runs).sort_values("finished", ascending=False) if runs else pd.DataFrame()
if runs.empty:
    raise SystemExit("No finished runs yet. Train one in Notebook 2 first.")
print(runs.to_string(index=False))

usable = runs[runs["cut as"] == crop["mode"]]
RUN_NAME = usable.iloc[0]["run"] if len(usable) else None   # or type a run name
if RUN_NAME is None:
    raise SystemExit(f"Notebook 1 last cut the pages as '{crop['mode']}', and no run was trained on those. "
                     "Train one in Notebook 2, or set CROP in Notebook 1 back to a run's kind and run its Steps 6 and 8.")
trained_as = runs.set_index("run").loc[RUN_NAME, "cut as"]
if trained_as != crop["mode"]:
    raise SystemExit(f"{RUN_NAME} was trained on '{trained_as}' images, but Notebook 1 last cut the pages as "
                     f"'{crop['mode']}'. Pick another run, or set CROP in Notebook 1 to '{trained_as}' "
                     "and run its Steps 6 and 8.")
print(f"\nTesting: {RUN_NAME}")

## Step 4. Load the model

This loads Qwen3-VL-4B with the Kurrent adapter and your trained adapter on top, onto the GPU. It
takes 1 to 3 minutes; the model is then ready for the rest of the notebook.

If you see `CUDA out of memory`, Notebook 2 or 3 may still be holding the GPU: in that notebook
choose Kernel → Shut Down Kernel, then run this cell again.

In [ ]:
import time
import torch
from finetune_qwen_kurrent_lora import load_for_inference, transcribe_lines
from kurrent_common import load_line_image

start = time.time()
model, processor = load_for_inference(os.path.join(RUNS, RUN_NAME, "model"), max_pixels=crop["max_pixels"])
print(f"Loaded in {time.time() - start:.0f}s on {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}.")

lines_dir = os.path.join(DATA, "kade_lines")

def read_page(page_id, before_training=False):
    """The model's transcription of one page: read each of its images, join them."""
    folder = os.path.join(lines_dir, page_id)
    names = sorted(n for n in os.listdir(folder) if n.endswith(".png")) if os.path.isdir(folder) else []
    images = [load_line_image(os.path.join(folder, n)) for n in names]
    if before_training:
        with model.disable_adapter():   # switch our adapter off: the Kurrent adapter on its own
            lines = transcribe_lines(model, processor, images, crop["max_new_tokens"], crop["eval_batch"])
    else:
        lines = transcribe_lines(model, processor, images, crop["max_new_tokens"], crop["eval_batch"])
    return "\n".join(lines)

## Step 5. Score pages

Choose which pages with `PAGES`:

- `"dev"`: the 11 dev pages (letter `kade_112`), which the model never trained on. **This is the
  fair test**, and the number to compare runs by.
- `"train"`: `HOW_MANY` pages the model trained on. These will look better than they really are,
  since it has seen them; useful only to check it learned anything at all.
- a list of page names, like `["kade_112_p004", "kade_128_p020"]`.

`COMPARE_BEFORE = True` also reads every page with the model as it was before training (takes
twice as long), so you see exactly what training changed.

About 20–60 seconds per page for lines on a g4dn, less on a g5. The results are saved in
`badger-finetune/scores/<run>/`.

In [ ]:
PAGES = "dev"          # "dev", "train", or a list of page names
HOW_MANY = 10          # for "train": how many pages
COMPARE_BEFORE = True  # also score the model before training

labels = pd.read_csv(os.path.join(DATA, "train.csv"), keep_default_na=False, encoding="utf-8-sig")
labels = labels.set_index("page_id")["text"]
if PAGES == "dev":
    page_ids = list(pd.read_csv(os.path.join(DATA, "dev_pages.csv"), keep_default_na=False)["page_id"])
elif PAGES == "train":
    trained = pd.read_csv(os.path.join(DATA, "train_lines.csv"), keep_default_na=False)["page_id"].unique()
    page_ids = sorted(pd.Series(trained).sample(min(HOW_MANY, len(trained)), random_state=0))
    print("Note: the model trained on these pages, so their scores look better than they really are.\n")
else:
    page_ids = list(PAGES)

rows = []
for i, pid in enumerate(page_ids, 1):
    start = time.time()
    row = {"page_id": pid, "reference": labels.get(pid, None), "after": read_page(pid)}
    if COMPARE_BEFORE:
        row["before"] = read_page(pid, before_training=True)
    if row["reference"] is not None:
        row["CER after"] = page_cer(row["after"], row["reference"])
        if COMPARE_BEFORE:
            row["CER before"] = page_cer(row["before"], row["reference"])
    rows.append(row)
    done = f"CER {row['CER after']:.3f}" if "CER after" in row else "no correct text to score against"
    if "CER before" in row:
        done += f" (before training {row['CER before']:.3f})"
    print(f"[{i}/{len(page_ids)}] {pid}: {done}, {time.time() - start:.0f}s", flush=True)

results = pd.DataFrame(rows)
name = PAGES if isinstance(PAGES, str) else "chosen"
out_dir = os.path.join(WORK, "scores", RUN_NAME)
os.makedirs(out_dir, exist_ok=True)
results.to_csv(os.path.join(out_dir, f"{name}.csv"), index=False)
print(f"\nSaved to {os.path.join(out_dir, name + '.csv')}")

## Step 6. The scores

CER is the share of characters wrong (0 is perfect, 1 is everything wrong); lower is better. The
mean is what Kaggle reports for the Kade letters. The reference text includes the `Postmark` line
some postcards have, as Kaggle's does, so those pages can't quite reach 0.

In [ ]:
scored = results[results["reference"].notna()] if "reference" in results else results.iloc[0:0]
if scored.empty:
    print("None of these pages has a correct transcription, so there's nothing to score. Use Step 7 to read them.")
else:
    cols = ["page_id"] + [c for c in ("CER before", "CER after") if c in scored]
    table = scored[cols].copy()
    if "CER before" in table:
        table["improvement"] = table["CER before"] - table["CER after"]
    print(table.round(3).to_string(index=False))
    print(f"\nMean CER after training:  {scored['CER after'].mean():.3f}  ({len(scored)} pages)")
    if "CER before" in scored:
        print(f"Mean CER before training: {scored['CER before'].mean():.3f}")
        better = int((table["improvement"] > 0).sum())
        print(f"Training helped on {better} of {len(table)} pages.")

## Step 7. Look at one page

The page image, then the text. In the comparison, <span style="background:#c8f7c5">green</span>
is what the model wrote that isn't in the correct text, and
<span style="background:#f7c5c5;text-decoration:line-through">red, struck through</span> is what it
missed or got wrong. The worst-scoring page is shown first; type any page name to see another,
including a test page (then you just get the model's reading).

In [ ]:
import difflib
import html
from IPython.display import HTML, display
from PIL import Image

if "CER after" in results and results["CER after"].notna().any():
    PAGE = results.sort_values("CER after", ascending=False).iloc[0]["page_id"]   # or type a page name
else:
    PAGE = results.iloc[0]["page_id"]

def coloured_diff(reference, prediction):
    out = []
    for op, a0, a1, b0, b1 in difflib.SequenceMatcher(None, reference, prediction, autojunk=False).get_opcodes():
        ref, pred = html.escape(reference[a0:a1]), html.escape(prediction[b0:b1])
        if op == "equal":
            out.append(ref)
        if op in ("delete", "replace"):
            out.append(f'<span style="background:#f7c5c5;text-decoration:line-through">{ref}</span>')
        if op in ("insert", "replace"):
            out.append(f'<span style="background:#c8f7c5">{pred}</span>')
    return "".join(out).replace("\n", "<br>")

row = results.set_index("page_id").loc[PAGE] if PAGE in set(results["page_id"]) else None
after = row["after"] if row is not None else read_page(PAGE)
reference = labels.get(PAGE, None)

image_path = os.path.join(DATA, "images", PAGE + ".jpg")
if os.path.exists(image_path):
    img = Image.open(image_path)
    img.thumbnail((600, 900))
    display(img)
else:
    print(f"There's no image for {PAGE} in {os.path.dirname(image_path)} (Notebook 1, Step 3).")
box = "font-family:monospace;white-space:normal;line-height:1.6;border:1px solid #ccc;padding:8px"
if reference is None:
    display(HTML(f"<h4>{PAGE}: the model's reading (no correct text to compare)</h4><div style='{box}'>"
                 + html.escape(after).replace("\n", "<br>") + "</div>"))
else:
    display(HTML(f"<h4>{PAGE}, after training: CER {page_cer(after, reference):.3f}</h4>"
                 f"<div style='{box}'>{coloured_diff(reference, after)}</div>"))
    if row is not None and "before" in row and isinstance(row["before"], str):
        display(HTML(f"<h4>{PAGE}, before training: CER {page_cer(row['before'], reference):.3f}</h4>"
                     f"<div style='{box}'>{coloured_diff(reference, row['before'])}</div>"))

## Done

- To test **another run**, change `RUN_NAME` in Step 3 and run from Step 3 again.
- When you've finished with this notebook, free the GPU for the others: **Kernel → Shut Down
  Kernel**.
- When you're done for the day, **stop the instance** (README.md, "Stopping and starting").